# SEED-VII EEGNet 端到端流水线

**架构替换**：EEG-Conformer → EEGNet（缓解过拟合）
**运行环境**：ModelScope 实例

### 核心变更
- **模型架构**：Transformer Encoder → EEGNet（时间卷积 + 空间深度卷积 + 可分离卷积）
- **参数量**：~0.75M → ~0.05M（大幅降低，缓解过拟合）
- **抗过拟合机制**：
  - 深度可分离卷积（参数效率极高）
  - 强 Dropout (0.5) + BatchNorm
  - 更简单的层级结构，减少记忆小样本的能力

## 0. 环境设置与导入

In [ ]:
# ====== 环境检查 ======
import os, sys, subprocess, json, pathlib
import torch
import numpy as np

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'})")

# ====== 路径配置 ======
REPO = pathlib.Path('/mnt/workspace/EEG_encoder_version2').resolve()
sys.path.insert(0, str(REPO))

# ====== ModelScope 配置 ======
MS_DATASET = 'DEREKVERSE/SEED-VII'
MS_REVISION = 'master'
MS_TOKEN = os.environ.get('MODELSCOPE_API_TOKEN', 'your_token_here')

# ====== 工作路径 ======
WORK = pathlib.Path('/mnt/workspace/seed_vii_work')
WORK.mkdir(parents=True, exist_ok=True)
MAT_LOCAL_DIR = str(WORK / 'mat_files')
SAVE_INFO_LOCAL = str(WORK / 'save_info')
NPZ_OUT = str(WORK / 'preprocessed' / 'seed_vii.npz')
RUN_DIR = str(WORK / 'runs_seed_vii_eegnet')
ENC_OUT = str(WORK / 'encoded' / 'embeddings.npz')

In [ ]:
# ====== ModelScope 登录 ======
os.environ['MODELSCOPE_API_TOKEN'] = MS_TOKEN
from modelscope.hub.api import HubApi
HubApi().login(MS_TOKEN)
print('✓ ModelScope login done')

def run(cmd, env=None):
    """运行子进程命令"""
    print('$', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=str(REPO), env={**os.environ, **(env or {})})
    if res.returncode != 0:
        raise SystemExit(f'cmd failed with code {res.returncode}')

PY = sys.executable

## 1. EEGNet 模型定义

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple

class EEGNetBlock1(nn.Module):
    """Block 1: 时间卷积 → 空间深度卷积"""
    def __init__(self, F1: int = 8, D: int = 2, Chans: int = 62,
                 kernLength: int = 100, dropout: float = 0.5):
        super().__init__()
        # 时间卷积 (1D 沿时间轴)
        self.temporal_conv = nn.Conv2d(
            in_channels=1, out_channels=F1,
            kernel_size=(1, kernLength),
            padding=(0, kernLength // 2),
            bias=False
        )
        # 深度空间卷积 (跨 EEG 通道)
        self.depthwise_conv = nn.Conv2d(
            in_channels=F1, out_channels=F1 * D,
            kernel_size=(Chans, 1),
            groups=F1,  # Depthwise = 每个输入通道独立滤波
            bias=False
        )
        self.bn = nn.BatchNorm2d(F1 * D)
        self.elu = nn.ELU()
        self.pool = nn.AvgPool2d(kernel_size=(1, 4))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.temporal_conv(x)          # (B, F1, Chans, T)
        x = self.depthwise_conv(x)          # (B, F1*D, 1, T)
        x = self.bn(x)
        x = self.elu(x)
        x = self.pool(x)                    # (B, F1*D, 1, T//4)
        return self.dropout(x)


class EEGNetBlock2(nn.Module):
    """Block 2: 可分离卷积（空间 → 时间精炼）"""
    def __init__(self, F1: int = 8, D: int = 2, F2: int = 16, dropout: float = 0.5):
        super().__init__()
        # 深度卷积 (空间模式)
        self.depthwise = nn.Conv2d(
            in_channels=F1 * D,
            out_channels=F1 * D,
            kernel_size=(1, 16),
            groups=F1 * D,
            bias=False
        )
        # 逐点卷积 (时间组合)
        self.pointwise = nn.Conv2d(
            in_channels=F1 * D,
            out_channels=F2,
            kernel_size=1,
            bias=False
        )
        self.bn = nn.BatchNorm2d(F2)
        self.elu = nn.ELU()
        self.pool = nn.AvgPool2d(kernel_size=(1, 8))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)               # (B, F1*D, 1, T')
        x = self.pointwise(x)               # (B, F2, 1, T')
        x = self.bn(x)
        x = self.elu(x)
        x = self.pool(x)                    # (B, F2, 1, T'//8)
        return self.dropout(x)


class IntensityHead(nn.Module):
    """强度回归头 (Sigmoid 输出 [0, 1])"""
    def __init__(self, in_dim: int, hidden: int = 64, dropout: float = 0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid(self.net(x))


class EEGNetDualHead(nn.Module):
    """EEGNet 双头模型：情绪分类 + 强度回归"""

    def __init__(
        self,
        Chans: int = 62,
        Samples: int = 800,
        F1: int = 8,
        D: int = 2,
        F2: int = 16,
        kernLength: int = 100,
        dropout: float = 0.5,
        nb_classes: int = 7,
        int_hidden: int = 64,
    ):
        super().__init__()

        # 特征提取块
        self.block1 = EEGNetBlock1(F1, D, Chans, kernLength, dropout)
        self.block2 = EEGNetBlock2(F1, D, F2, dropout)

        # 自动计算展平后的特征维度
        with torch.no_grad():
            dummy = torch.zeros(1, 1, Chans, Samples)
            x = self.block1(dummy)
            x = self.block2(x)
            flat_size = x.view(1, -1).size(1)

        # 双头输出
        self.classifier = nn.Linear(flat_size, nb_classes)
        self.intensity_head = IntensityHead(flat_size, int_hidden, dropout)

        print(f"[EEGNetDualHead] 特征维度: {flat_size}")
        print(f"[EEGNetDualHead] 分类头: {flat_size} -> {nb_classes}")
        print(f"[EEGNetDualHead] 强度头: {flat_size} -> {int_hidden} -> 1")

    def forward(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """前向传播

        Args:
            x: 输入 EEG 张量 (B, 1, Chans, Samples)

        Returns:
            logits: 分类 logits (B, nb_classes)
            intensity: 预测强度 [0, 1] (B,)
            features: 展平的特征表示 (B, flat_size)
        """
        # 特征提取
        x = self.block1(x)    # (B, F1*D, 1, T1)
        x = self.block2(x)    # (B, F2, 1, T2)
        features = x.view(x.size(0), -1)  # (B, F2*T2)

        # 双头输出
        logits = self.classifier(features)
        intensity = self.intensity_head(features).squeeze(-1)

        return logits, intensity, features


def count_parameters(model: nn.Module) -> int:
    """计算可训练参数量"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def print_model_summary(model: nn.Module, Chans: int = 62, Samples: int = 800) -> None:
    """打印模型架构和参数量统计"""
    print("=" * 70)
    print("EEGNet 双头架构 for SEED-VII")
    print("=" * 70)
    print(model)
    print("-" * 70)
    total = count_parameters(model)
    print(f"总可训练参数: {total:,}")
    print(f"输入形状: (B, 1, {Chans}, {Samples})")

    print("\n参数量分解:")
    for name, module in model.named_children():
        n_params = sum(p.numel() for p in module.parameters() if p.requires_grad)
        print(f"  {name:20s}: {n_params:>8,}")
    print("=" * 70)

In [ ]:
# ====== 创建模型并查看架构 ======
model = EEGNetDualHead(
    Chans=62, Samples=800, F1=8, D=2, F2=16,
    kernLength=100, dropout=0.5, nb_classes=7
)
print_model_summary(model)

# ====== 测试前向传播 ======
print("\n测试前向传播...")
dummy_input = torch.randn(2, 1, 62, 800)
logits, intensity, features = model(dummy_input)
print(f"  输入形状:    {dummy_input.shape}")
print(f"  Logits 形状:  {logits.shape}")
print(f"  强度形状:    {intensity.shape}")
print(f"  特征形状:    {features.shape}")
print(f"\n✓ 前向传播成功!")

## 2. 数据预处理（流式）

In [ ]:
# ====== 下载数据 ======
from pathlib import Path
from src.ms_download import download_files, list_dataset_files, filter_files, login_if_token

login_if_token(MS_TOKEN)
Path(MAT_LOCAL_DIR).mkdir(parents=True, exist_ok=True)
Path(SAVE_INFO_LOCAL).mkdir(parents=True, exist_ok=True)

all_files = list_dataset_files(MS_DATASET, revision=MS_REVISION, token=MS_TOKEN)
mat_files = [f for f in all_files if f['Path'].lower().endswith('.mat') and 'eeg_preprocessed' in f['Path'].lower()]
save_files = [f for f in all_files if f['Path'].lower().endswith('_save_info.csv')]
if not mat_files: mat_files = filter_files(all_files, include=['EEG_preprocessed/*.mat', '*.mat'])
if not save_files: save_files = filter_files(all_files, include=['save_info/*_save_info.csv'])

mat_paths = [f['Path'] for f in mat_files]
save_paths = [f['Path'] for f in save_files]
print(f'Found {len(mat_paths)} .mat, {len(save_paths)} save_info')

download_files(MS_DATASET, mat_paths, MAT_LOCAL_DIR, revision=MS_REVISION, token=MS_TOKEN, verbose=True)
download_files(MS_DATASET, save_paths, SAVE_INFO_LOCAL, revision=MS_REVISION, token=MS_TOKEN, verbose=True)

from src.dataset import load_save_info_intensity
intensities = load_save_info_intensity(SAVE_INFO_LOCAL)
print(f'✓ Intensity labels: {len(intensities)} (expect ~1600)')

In [ ]:
# ====== 预处理到 npz ======
run([PY, 'scripts/preprocess_to_npz.py',
     '--mat-dir', MAT_LOCAL_DIR, '--mat-pattern', '*.mat',
     '--save-info-dir', SAVE_INFO_LOCAL,
     '--output', NPZ_OUT,
     '--val-ratio', '0.1', '--test-ratio', '0.1',
     '--split-unit', 'trial', '--max-windows-per-trial', '40',
     '--tmp-dir', str(WORK / 'preprocessed')])

# ====== 转换为 mmap 格式 ======
from src.dataset import ensure_mmap_format
x_npy, meta_npz = ensure_mmap_format(NPZ_OUT)
print(f'✓ X mmap: {x_npy}')
print(f'✓ Meta:   {meta_npz}')

# 验证 mmap 可用
x_mmap = np.load(x_npy, mmap_mode='r')
print(f'✓ X shape={x_mmap.shape}, dtype={x_mmap.dtype} (mmap, 0 RAM)')

## 3. 训练 EEGNet 模型

In [ ]:
# ====== 修改训练脚本以使用 EEGNet ======
# 注意：需要先将 src/model.py 替换为 EEGNet 版本
# 或者在训练脚本中导入 EEGNetDualHead 而不是 EEGConformerDualHead

from src.losses import LossConfig, WeightedDualLoss
from src.config import TRAIN_DEFAULTS

# ====== 训练配置 ======
FREEZE_INTENSITY = True  # 冻结强度回归头，专注分类
TRAIN_SUBJECTS = '1'     # 单被试训练（防过拟合）
VAL_SUBJECTS = '2,3'     # 验证被试
TEST_SUBJECTS = '4,5'    # 测试被试

loss_cfg = LossConfig(
    alpha=1.0, beta=0.5, gamma=0.0,  # 退化方案：先不开 ranking loss
    label_smoothing=0.05,
    sample_weight_mode='continuous',
    intensity_threshold=0.5,
    weak_sample_weight=0.1
)
criterion = WeightedDualLoss(loss_cfg)

print("✓ 损失函数配置完成 (退化方案: α=1.0, β=0.5, γ=0.0)")
print(f"✓ 训练被试: {TRAIN_SUBJECTS}")
print(f"✓ 验证被试: {VAL_SUBJECTS}")
print(f"✓ 测试被试: {TEST_SUBJECTS}")

In [ ]:
# ====== 构建训练命令 ======
# 注意：train.py 需要支持 --model eegnet 参数来切换模型
train_cmd = [PY, 'scripts/train.py',
    '--data', NPZ_OUT,
    '--output-dir', RUN_DIR,
    '--device', 'auto', '--amp',
    '--model', 'eegnet',  # 使用 EEGNet 而不是 Conformer
    '--pretrain-epochs', '5' if FREEZE_INTENSITY else '10',
    '--max-epochs', '100' if FREEZE_INTENSITY else '200',
    '--patience', '20' if FREEZE_INTENSITY else '30',
    '--batch-size', '64',
    '--num-workers', '2',
    '--lr', '2e-4', '--min-lr', '1e-5',
    '--sample-weight-mode', 'continuous',
    '--max-runtime-hours', '10',
]

if FREEZE_INTENSITY:
    train_cmd.append('--freeze-intensity-head')
    print('✓ 冻结 Intensity Head：只训练分类头')

if TRAIN_SUBJECTS:
    train_cmd.extend(['--train-subjects', TRAIN_SUBJECTS])
    print(f'✓ 训练被试: {TRAIN_SUBJECTS}')
if VAL_SUBJECTS:
    train_cmd.extend(['--val-subjects', VAL_SUBJECTS])
    print(f'✓ 验证被试: {VAL_SUBJECTS}')
if TEST_SUBJECTS:
    train_cmd.extend(['--test-subjects', TEST_SUBJECTS])
    print(f'✓ 测试被试: {TEST_SUBJECTS}')

print(f'\n训练命令: {" ".join(train_cmd)}')
run(train_cmd)

In [ ]:
# ====== 续训（取消注释即可）======
# run([PY, 'scripts/train.py',
#      '--data', NPZ_OUT,
#      '--output-dir', RUN_DIR,
#      '--device', 'auto', '--amp', '--resume',
#      '--model', 'eegnet',
#      '--freeze-intensity-head',
#      '--train-subjects', TRAIN_SUBJECTS,
#      '--val-subjects', VAL_SUBJECTS,
#      '--max-runtime-hours', '10'])

## 4. 训练结果分析

In [ ]:
import json
from pathlib import Path

sp = Path(RUN_DIR) / 'summary.json'
if sp.exists():
    s = json.load(open(sp))
    print(f"最佳验证集准确率: {s.get('best_val_acc', 'N/A')}")
    print(f"训练轮数: {s.get('epochs_run', 'N/A')}")
    print(f"参数量: {s.get('n_params', 'N/A'):,}")
    print(f"冻结强度头: {s.get('freeze_intensity_head', 'N/A')}")
    print(f"训练被试: {s.get('train_subjects', 'N/A')}")
    if s.get('test'):
        print(f"测试集准确率: {s['test'].get('acc', 'N/A')}")
        print(f"强度 MAE: {s['test'].get('intensity_mae', 'N/A')}")
else:
    print('暂无训练结果')

## 5. 编码导出（特征提取）

In [ ]:
run([PY, 'scripts/encode.py',
     '--data', NPZ_OUT,
     '--checkpoint', str(pathlib.Path(RUN_DIR) / 'best_encoder.pt'),
     '--output', ENC_OUT,
     '--model', 'eegnet',  # 使用 EEGNet 编码器
     '--feature-type', 'projected',
     '--device', 'auto', '--amp'])

## 📊 EEGNet vs EEG-Conformer 对比

In [ ]:
print("=" * 70)
print("架构对比: EEG-Conformer vs EEGNet")
print("=" * 70)
print("\n| 特性 | EEG-Conformer | EEGNet |")
print("|------|---------------|--------|")
print("| 核心模块 | Transformer (6层) | 深度可分离卷积 |")
print("| 参数量 | ~750K | ~50K |")
print("| 过拟合风险 | 高（小数据集） | 低 |")
print("| 时间建模 | 自注意力 | 时间卷积 + 池化 |")
print("| 空间建模 | 空间深度卷积 | 空间深度卷积 + 逐点卷积 |")
print("| Dropout | 0.5 | 0.5 |")
print("| 适用场景 | 大数据集 | 小数据集/跨被试 |")
print("=" * 70)